# 01 Data Cleaning And Wrangling

The raw file is intentionally messy. This notebook shows the cleaning logic used before analysis.

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv('../data/raw/food_delivery_raw.csv')
raw.head()

In [ ]:
raw.shape, raw.duplicated('order_id').sum(), raw.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df = raw.copy()
df['city'] = df['city'].astype(str).str.strip().str.title().replace({'Delhi-Ncr':'Delhi NCR'})
for col in ['zone','cuisine_type','payment_method','weather_condition']:
    df[col] = df[col].astype(str).str.strip().str.title().replace({'':'Unknown','Nan':'Unknown'})
df['order_datetime'] = pd.to_datetime(df['order_datetime'], errors='coerce', dayfirst=True)
df = df.drop_duplicates('order_id', keep='last')
df['delivery_partner_id'] = df['delivery_partner_id'].fillna('UNASSIGNED').replace('', 'UNASSIGNED')
df['discount_amount'] = pd.to_numeric(df['discount_amount'], errors='coerce').fillna(0)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(pd.to_numeric(df['rating'], errors='coerce').median())
for col in ['delivery_time_minutes','promised_delivery_time_minutes','order_value','delivery_fee','packaging_charge','platform_commission','refund_amount','distance_km']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['delivery_time_minutes'] = df['delivery_time_minutes'].clip(10, 180)
df['delivery_delay_flag'] = (df['delivery_time_minutes'] > df['promised_delivery_time_minutes']).astype(int)
df['gross_profit'] = df['platform_commission'] + df['delivery_fee'] + df['packaging_charge'] - df['discount_amount'] - df['refund_amount']
df['profit_margin_pct'] = (df['gross_profit'] / df['order_value'] * 100).round(2)
df['delay_minutes'] = (df['delivery_time_minutes'] - df['promised_delivery_time_minutes']).clip(lower=0)
df['discount_rate'] = df['discount_amount'] / df['order_value']
df['discount_band'] = pd.cut(df['discount_rate'], [-0.01,0,0.10,0.20,1], labels=['No Discount','Low','Medium','High'])
df['order_month'] = df['order_datetime'].dt.to_period('M').astype(str)
df.to_csv('../data/processed/food_delivery_cleaned.csv', index=False)
df.head()